### 05_embeddings.ipynb

In [1]:
!pip install sentence-transformers qdrant-client

In [2]:
# BLOCK 1 — IMPORT LIBRARIES

import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer

from sklearn.metrics.pairwise import cosine_similarity

from qdrant_client import QdrantClient

from qdrant_client.models import (
    Distance,
    VectorParams,
    PointStruct
)

import warnings
warnings.filterwarnings("ignore")

In [3]:
# USE PARQUET # Faster and cleaner than CSV 
df = pd.read_parquet( "../data/clean/ml_merged_dataset.parquet" ) 
print("Dataset Loaded Successfully") 
print("\nDataset Shape:\n") 
print(df.shape) 
print("\nFirst 5 Rows:\n") 
display(df.head()) 
print("\nColumns:\n") 
print(df.columns.tolist())

Dataset Loaded Successfully

Dataset Shape:

(180519, 27)

First 5 Rows:



,order_id,customer_id,product_name,category,department,Product Price,product_image,Order Region,Market,Order Status,...,Longitude,delay,view_count,unique_users,peak_month,peak_hour,node_id,degree,out_degree,in_degree
0,77202,20755,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,Southeast Asia,Pacific Asia,COMPLETE,...,-66.037056,-1.0,0.0,0.0,Sep,21.0,0,0.0,0.0,0.0
1,75939,19492,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,South Asia,Pacific Asia,PENDING,...,-66.037064,1.0,0.0,0.0,Sep,21.0,1,3.0,3.0,0.0
2,75938,19491,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,South Asia,Pacific Asia,CLOSED,...,-121.881279,0.0,0.0,0.0,Sep,21.0,2,5.0,5.0,0.0
3,75937,19490,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,Oceania,Pacific Asia,COMPLETE,...,-118.291016,-1.0,0.0,0.0,Sep,21.0,3,0.0,0.0,0.0
4,75936,19489,smart watch,sporting goods,Fitness,327.75,http://images.acmesports.sports/Smart+watch,Oceania,Pacific Asia,PENDING_PAYMENT,...,-66.037048,-2.0,0.0,0.0,Sep,21.0,4,6.0,6.0,0.0



Columns:

['order_id', 'customer_id', 'product_name', 'category', 'department', 'Product Price', 'product_image', 'Order Region', 'Market', 'Order Status', 'Shipping Mode', 'actual_days', 'scheduled_days', 'sales', 'quantity', 'profit', 'Latitude', 'Longitude', 'delay', 'view_count', 'unique_users', 'peak_month', 'peak_hour', 'node_id', 'degree', 'out_degree', 'in_degree']


In [4]:
# BLOCK 4 — CREATE TEXT FIELD 
df["text_data"] = ( df["product_name"].fillna("").astype(str) + " " + df["category"].fillna("").astype(str) + " " + df["department"].fillna("").astype(str) )

In [5]:
# Keep only required columns
df = df[[ "node_id", "text_data" ]] 
# Remove empty rows 
df = df[ df["text_data"].str.strip() != "" ]

In [6]:
# Reset index 
df = df.reset_index(drop=True) 
print("Text Data Created Successfully") 
print("\nDataset Shape:\n") 
print(df.shape) 
display(df.head())

Text Data Created Successfully

Dataset Shape:

(180519, 2)


,node_id,text_data
0,0,smart watch sporting goods Fitness
1,1,smart watch sporting goods Fitness
2,2,smart watch sporting goods Fitness
3,3,smart watch sporting goods Fitness
4,4,smart watch sporting goods Fitness


In [7]:
# BLOCK 5 — LOAD SENTENCE TRANSFORMER MODEL
model = SentenceTransformer( "all-MiniLM-L6-v2" ) 
print("SentenceTransformer Model Loaded Successfully")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

SentenceTransformer Model Loaded Successfully


In [8]:
# BLOCK 6 — GENERATE EMBEDDINGS
texts = df["text_data"].tolist() 
embeddings = model.encode( texts, show_progress_bar=True ) 
print("Embeddings Generated Successfully") 
print("\nEmbedding Shape:\n") 
print(embeddings.shape)

Batches:   0%|          | 0/5642 [00:00<?, ?it/s]

Embeddings Generated Successfully

Embedding Shape:

(180519, 384)


In [9]:
# BLOCK 7 — CONNECT TO QDRANT

client = QdrantClient(":memory:")

print("Connected to Local Qdrant Successfully")

Connected to Local Qdrant Successfully


In [10]:
# BLOCK 8 — CREATE COLLECTION
COLLECTION_NAME = "capstone_embeddings"

# Delete old collection if it already exists

if client.collection_exists(COLLECTION_NAME):

    client.delete_collection(COLLECTION_NAME)

    print("Old Collection Deleted")


# Create new collection

client.create_collection(

    collection_name=COLLECTION_NAME,

    vectors_config=VectorParams(

        size=384,

        distance=Distance.COSINE
    )
)

print("Qdrant Collection Created Successfully")

Qdrant Collection Created Successfully


In [11]:
# BLOCK 9 — CREATE QDRANT POINTS
points = []

for idx, row in df.iterrows():

    point = PointStruct(

        id=int(idx),

        vector=embeddings[idx].tolist(),

        payload={

            "node_id": str(row["node_id"]),

            "raw_text": row["text_data"]
        }
    )

    points.append(point)

print("Points Created Successfully")

print("\nTotal Points:\n")

print(len(points))

Points Created Successfully

Total Points:

180519


In [12]:
# BLOCK 10 — UPSERT EMBEDDINGS TO QDRANT
# Batch upload for better performance

BATCH_SIZE = 1000

for i in range(0, len(points), BATCH_SIZE):

    batch = points[i:i+BATCH_SIZE]

    client.upsert(

        collection_name=COLLECTION_NAME,

        points=batch
    )

print("Embeddings Uploaded Successfully")

Embeddings Uploaded Successfully


In [13]:
# BLOCK 11 — SIMILARITY SEARCH QUERY
query_text = "running shoes for fitness training"

print("\nQuery Text:\n")

print(query_text)


Query Text:

running shoes for fitness training


In [14]:
# BLOCK 12 — EMBED QUERY

query_vector = model.encode(
    query_text
).tolist()

print("Query Embedding Generated Successfully")

Query Embedding Generated Successfully


In [15]:
# BLOCK 13 — QUERY QDRANT
results = client.query_points(

    collection_name=COLLECTION_NAME,

    query=query_vector,

    limit=5,

    with_payload=True
)

print("Similarity Search Completed Successfully")

Similarity Search Completed Successfully


In [16]:
# BLOCK 14 — DISPLAY RESULTS
print("\nTOP 5 SIMILAR RESULTS\n")

for idx, hit in enumerate(results.points, start=1):

    print("=" * 70)

    print(f"RESULT {idx}")

    print(f"\nSimilarity Score: {round(hit.score, 4)}")

    print(f"\nNode ID: {hit.payload['node_id']}")

    print("\nRetrieved Text:")

    print(hit.payload["raw_text"])

    print("=" * 70)


TOP 5 SIMILAR RESULTS

RESULT 1

Similarity Score: 0.6172

Node ID: 47232

Retrieved Text:
nike men's free 5.0+ running shoe cardio equipment Footwear
RESULT 2

Similarity Score: 0.6172

Node ID: 28825

Retrieved Text:
nike men's free 5.0+ running shoe cardio equipment Footwear
RESULT 3

Similarity Score: 0.6172

Node ID: 165146

Retrieved Text:
nike men's free 5.0+ running shoe cardio equipment Footwear
RESULT 4

Similarity Score: 0.6172

Node ID: 165145

Retrieved Text:
nike men's free 5.0+ running shoe cardio equipment Footwear
RESULT 5

Similarity Score: 0.6172

Node ID: 60837

Retrieved Text:
nike men's free 5.0+ running shoe cardio equipment Footwear


In [17]:
#BLOCK 15 — SAVE EMBEDDINGS
embedding_df = pd.DataFrame(embeddings)

# Add node_id for traceability

embedding_df["node_id"] = df["node_id"]

embedding_df.to_parquet(
    "../data/clean/product_embeddings.parquet",
    index=False
)

print("Embeddings Saved Successfully")

Embeddings Saved Successfully


In [18]:
# BLOCK 16 — SAVE FINAL DATASET
df.to_parquet(
    "../data/clean/embedding_dataset.parquet",
    index=False
)

print("Embedding Dataset Saved Successfully")

Embedding Dataset Saved Successfully


In [19]:
#BLOCK 17 — FINAL SUMMARY
print("\n================================================")

print("05_EMBEDDINGS NOTEBOOK COMPLETED SUCCESSFULLY")

print("================================================")

print("\nOPERATIONS COMPLETED:")

print("- Loaded Dataset")
print("- Created Text Field")
print("- Generated Sentence Embeddings")
print("- Connected to Qdrant")
print("- Created Collection")
print("- Uploaded Embeddings")
print("- Performed Similarity Search")

print("\nMODEL USED:")

print("- all-MiniLM-L6-v2")

print("\nVECTOR DATABASE:")

print("- Qdrant")
print("- COSINE Similarity")
print("- 384-dimensional vectors")

print("\nRAG STATUS:")

print("- Retrieval COMPLETE")
print("- Augmentation READY")
print("- Generation conceptual only")


05_EMBEDDINGS NOTEBOOK COMPLETED SUCCESSFULLY

OPERATIONS COMPLETED:
- Loaded Dataset
- Created Text Field
- Generated Sentence Embeddings
- Connected to Qdrant
- Created Collection
- Uploaded Embeddings
- Performed Similarity Search

MODEL USED:
- all-MiniLM-L6-v2

VECTOR DATABASE:
- Qdrant
- COSINE Similarity
- 384-dimensional vectors

RAG STATUS:
- Retrieval COMPLETE
- Augmentation READY
- Generation conceptual only


In [22]:
print(df.columns)

Index(['node_id', 'text_data'], dtype='object')


In [23]:
# ============================================================
# CREATE FINAL EMBEDDINGS DATAFRAME
# ============================================================

embeddings_df = pd.DataFrame({

    "node_id": df["node_id"],

    "raw_text": df["text_data"],

    "embedding": embeddings.tolist()
})

# ============================================================
# SAVE FINAL PARQUET
# ============================================================

embeddings_df.to_parquet(
    "../data/clean/product_embeddings.parquet",
    index=False
)

print("Embeddings parquet saved successfully.")

Embeddings parquet saved successfully.
